# Naive RAG 구현 준비

네이버 2026년도 1분기 실적 발표 PDF를 사용해서 가장 기본적인 RAG 파이프라인을 만들어봅니다.

흐름:

1. Document Loader로 PDF 문서 불러오기
2. Text Splitter로 문서 청크 분할하기
3. Embedding 적용하기
4. Vector DB 생성하기
5. Retriever로 관련 문서 검색하기
6. 검색된 문서를 기반으로 LLM 답변 생성하기

## 0. 환경 준비

`.env` 파일에 `OPENAI_API_KEY`가 들어있어야 합니다.

In [1]:
from __future__ import annotations

import os
from pathlib import Path

from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv()

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PDF_PATH = PROJECT_ROOT / "docs" / "social_culture_doc.pdf"
DB_DIR = PROJECT_ROOT / "data" / "chroma_social_culture"
COLLECTION_NAME = "social_culture"

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"PDF exists: {PDF_PATH.exists()} - {PDF_PATH}")
print(f"OPENAI_API_KEY set: {bool(os.getenv('OPENAI_API_KEY'))}")

/Users/judy/Desktop/github/mo_du_yeon/Advanced-RAG-Multimodal-AI-Agent/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PROJECT_ROOT: /Users/judy/Desktop/github/mo_du_yeon/Advanced-RAG-Multimodal-AI-Agent
PDF exists: True - /Users/judy/Desktop/github/mo_du_yeon/Advanced-RAG-Multimodal-AI-Agent/docs/social_culture_doc.pdf
OPENAI_API_KEY set: True


In [2]:
from __future__ import annotations

import importlib.util
import sys
import os
import re
import shutil
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from PIL import Image, ImageDraw

print(sys.executable)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
IMG_DIR = PROJECT_ROOT / "docs" / "imgs"

# 외부 라이브러리가 홈 디렉터리에 캐시를 만들지 않도록 프로젝트 내부로 고정합니다.
os.environ.setdefault("PADDLE_PDX_CACHE_HOME", str(PROJECT_ROOT / "data" / "paddlex_cache"))
os.environ.setdefault("MPLCONFIGDIR", str(PROJECT_ROOT / "data" / "matplotlib_cache"))
os.environ.setdefault("XDG_CACHE_HOME", str(PROJECT_ROOT / "data" / "xdg_cache"))
for key in ["PADDLE_PDX_CACHE_HOME", "MPLCONFIGDIR", "XDG_CACHE_HOME"]:
    Path(os.environ[key]).mkdir(parents=True, exist_ok=True)


def has_module(name: str) -> bool:
    return importlib.util.find_spec(name) is not None


for module in ["pytesseract", "easyocr", "paddleocr", "cv2", "layoutparser"]:
    print(f"{module:14s}: {has_module(module)}")
print("tesseract bin :", shutil.which("tesseract"))

/Users/judy/Desktop/github/mo_du_yeon/Advanced-RAG-Multimodal-AI-Agent/venv/bin/python
pytesseract   : True
easyocr       : True
paddleocr     : True
cv2           : True
layoutparser  : True
tesseract bin : /opt/homebrew/bin/tesseract


In [ ]:
from __future__ import annotations

import importlib.util
import sys
import os
import re
import shutil
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from PIL import Image, ImageDraw

print(sys.executable)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
IMG_DIR = PROJECT_ROOT / "docs" / "imgs"

# 외부 라이브러리가 홈 디렉터리에 캐시를 만들지 않도록 프로젝트 내부로 고정합니다.
os.environ.setdefault("PADDLE_PDX_CACHE_HOME", str(PROJECT_ROOT / "data" / "paddlex_cache"))
os.environ.setdefault("MPLCONFIGDIR", str(PROJECT_ROOT / "data" / "matplotlib_cache"))
os.environ.setdefault("XDG_CACHE_HOME", str(PROJECT_ROOT / "data" / "xdg_cache"))
for key in ["PADDLE_PDX_CACHE_HOME", "MPLCONFIGDIR", "XDG_CACHE_HOME"]:
    Path(os.environ[key]).mkdir(parents=True, exist_ok=True)


def has_module(name: str) -> bool:
    return importlib.util.find_spec(name) is not None


for module in ["pytesseract", "easyocr", "paddleocr", "cv2", "layoutparser"]:
    print(f"{module:14s}: {has_module(module)}")
print("tesseract bin :", shutil.which("tesseract"))

/Users/judy/Desktop/github/mo_du_yeon/Advanced-RAG-Multimodal-AI-Agent/venv/bin/python
pytesseract   : True
easyocr       : True
paddleocr     : True
cv2           : True
layoutparser  : True
tesseract bin : /opt/homebrew/bin/tesseract


In [ ]:
from __future__ import annotations

import importlib.util
import sys
import os
import re
import shutil
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from PIL import Image, ImageDraw

print(sys.executable)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
IMG_DIR = PROJECT_ROOT / "docs" / "imgs"

# 외부 라이브러리가 홈 디렉터리에 캐시를 만들지 않도록 프로젝트 내부로 고정합니다.
os.environ.setdefault("PADDLE_PDX_CACHE_HOME", str(PROJECT_ROOT / "data" / "paddlex_cache"))
os.environ.setdefault("MPLCONFIGDIR", str(PROJECT_ROOT / "data" / "matplotlib_cache"))
os.environ.setdefault("XDG_CACHE_HOME", str(PROJECT_ROOT / "data" / "xdg_cache"))
for key in ["PADDLE_PDX_CACHE_HOME", "MPLCONFIGDIR", "XDG_CACHE_HOME"]:
    Path(os.environ[key]).mkdir(parents=True, exist_ok=True)


def has_module(name: str) -> bool:
    return importlib.util.find_spec(name) is not None


for module in ["pytesseract", "easyocr", "paddleocr", "cv2", "layoutparser"]:
    print(f"{module:14s}: {has_module(module)}")
print("tesseract bin :", shutil.which("tesseract"))

/Users/judy/Desktop/github/mo_du_yeon/Advanced-RAG-Multimodal-AI-Agent/venv/bin/python
pytesseract   : True
easyocr       : True
paddleocr     : True
cv2           : True
layoutparser  : True
tesseract bin : /opt/homebrew/bin/tesseract


In [ ]:
from __future__ import annotations

import importlib.util
import sys
import os
import re
import shutil
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from PIL import Image, ImageDraw

print(sys.executable)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
IMG_DIR = PROJECT_ROOT / "docs" / "imgs"

# 외부 라이브러리가 홈 디렉터리에 캐시를 만들지 않도록 프로젝트 내부로 고정합니다.
os.environ.setdefault("PADDLE_PDX_CACHE_HOME", str(PROJECT_ROOT / "data" / "paddlex_cache"))
os.environ.setdefault("MPLCONFIGDIR", str(PROJECT_ROOT / "data" / "matplotlib_cache"))
os.environ.setdefault("XDG_CACHE_HOME", str(PROJECT_ROOT / "data" / "xdg_cache"))
for key in ["PADDLE_PDX_CACHE_HOME", "MPLCONFIGDIR", "XDG_CACHE_HOME"]:
    Path(os.environ[key]).mkdir(parents=True, exist_ok=True)


def has_module(name: str) -> bool:
    return importlib.util.find_spec(name) is not None


for module in ["pytesseract", "easyocr", "paddleocr", "cv2", "layoutparser"]:
    print(f"{module:14s}: {has_module(module)}")
print("tesseract bin :", shutil.which("tesseract"))

/Users/judy/Desktop/github/mo_du_yeon/Advanced-RAG-Multimodal-AI-Agent/venv/bin/python
pytesseract   : True
easyocr       : True
paddleocr     : True
cv2           : True
layoutparser  : True
tesseract bin : /opt/homebrew/bin/tesseract


In [ ]:
from __future__ import annotations

import importlib.util
import sys
import os
import re
import shutil
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from PIL import Image, ImageDraw

print(sys.executable)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
IMG_DIR = PROJECT_ROOT / "docs" / "imgs"

# 외부 라이브러리가 홈 디렉터리에 캐시를 만들지 않도록 프로젝트 내부로 고정합니다.
os.environ.setdefault("PADDLE_PDX_CACHE_HOME", str(PROJECT_ROOT / "data" / "paddlex_cache"))
os.environ.setdefault("MPLCONFIGDIR", str(PROJECT_ROOT / "data" / "matplotlib_cache"))
os.environ.setdefault("XDG_CACHE_HOME", str(PROJECT_ROOT / "data" / "xdg_cache"))
for key in ["PADDLE_PDX_CACHE_HOME", "MPLCONFIGDIR", "XDG_CACHE_HOME"]:
    Path(os.environ[key]).mkdir(parents=True, exist_ok=True)


def has_module(name: str) -> bool:
    return importlib.util.find_spec(name) is not None


for module in ["pytesseract", "easyocr", "paddleocr", "cv2", "layoutparser"]:
    print(f"{module:14s}: {has_module(module)}")
print("tesseract bin :", shutil.which("tesseract"))

/Users/judy/Desktop/github/mo_du_yeon/Advanced-RAG-Multimodal-AI-Agent/venv/bin/python
pytesseract   : True
easyocr       : True
paddleocr     : True
cv2           : True
layoutparser  : True
tesseract bin : /opt/homebrew/bin/tesseract


In [ ]:
from __future__ import annotations

import importlib.util
import sys
import os
import re
import shutil
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from PIL import Image, ImageDraw

print(sys.executable)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
IMG_DIR = PROJECT_ROOT / "docs" / "imgs"

# 외부 라이브러리가 홈 디렉터리에 캐시를 만들지 않도록 프로젝트 내부로 고정합니다.
os.environ.setdefault("PADDLE_PDX_CACHE_HOME", str(PROJECT_ROOT / "data" / "paddlex_cache"))
os.environ.setdefault("MPLCONFIGDIR", str(PROJECT_ROOT / "data" / "matplotlib_cache"))
os.environ.setdefault("XDG_CACHE_HOME", str(PROJECT_ROOT / "data" / "xdg_cache"))
for key in ["PADDLE_PDX_CACHE_HOME", "MPLCONFIGDIR", "XDG_CACHE_HOME"]:
    Path(os.environ[key]).mkdir(parents=True, exist_ok=True)


def has_module(name: str) -> bool:
    return importlib.util.find_spec(name) is not None


for module in ["pytesseract", "easyocr", "paddleocr", "cv2", "layoutparser"]:
    print(f"{module:14s}: {has_module(module)}")
print("tesseract bin :", shutil.which("tesseract"))

/Users/judy/Desktop/github/mo_du_yeon/Advanced-RAG-Multimodal-AI-Agent/venv/bin/python
pytesseract   : True
easyocr       : True
paddleocr     : True
cv2           : True
layoutparser  : True
tesseract bin : /opt/homebrew/bin/tesseract


In [ ]:
from __future__ import annotations

import importlib.util
import sys
import os
import re
import shutil
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from PIL import Image, ImageDraw

print(sys.executable)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
IMG_DIR = PROJECT_ROOT / "docs" / "imgs"

# 외부 라이브러리가 홈 디렉터리에 캐시를 만들지 않도록 프로젝트 내부로 고정합니다.
os.environ.setdefault("PADDLE_PDX_CACHE_HOME", str(PROJECT_ROOT / "data" / "paddlex_cache"))
os.environ.setdefault("MPLCONFIGDIR", str(PROJECT_ROOT / "data" / "matplotlib_cache"))
os.environ.setdefault("XDG_CACHE_HOME", str(PROJECT_ROOT / "data" / "xdg_cache"))
for key in ["PADDLE_PDX_CACHE_HOME", "MPLCONFIGDIR", "XDG_CACHE_HOME"]:
    Path(os.environ[key]).mkdir(parents=True, exist_ok=True)


def has_module(name: str) -> bool:
    return importlib.util.find_spec(name) is not None


for module in ["pytesseract", "easyocr", "paddleocr", "cv2", "layoutparser"]:
    print(f"{module:14s}: {has_module(module)}")
print("tesseract bin :", shutil.which("tesseract"))

/Users/judy/Desktop/github/mo_du_yeon/Advanced-RAG-Multimodal-AI-Agent/venv/bin/python
pytesseract   : True
easyocr       : True
paddleocr     : True
cv2           : True
layoutparser  : True
tesseract bin : /opt/homebrew/bin/tesseract


In [ ]:
from __future__ import annotations

import importlib.util
import sys
import os
import re
import shutil
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from PIL import Image, ImageDraw

print(sys.executable)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
IMG_DIR = PROJECT_ROOT / "docs" / "imgs"

# 외부 라이브러리가 홈 디렉터리에 캐시를 만들지 않도록 프로젝트 내부로 고정합니다.
os.environ.setdefault("PADDLE_PDX_CACHE_HOME", str(PROJECT_ROOT / "data" / "paddlex_cache"))
os.environ.setdefault("MPLCONFIGDIR", str(PROJECT_ROOT / "data" / "matplotlib_cache"))
os.environ.setdefault("XDG_CACHE_HOME", str(PROJECT_ROOT / "data" / "xdg_cache"))
for key in ["PADDLE_PDX_CACHE_HOME", "MPLCONFIGDIR", "XDG_CACHE_HOME"]:
    Path(os.environ[key]).mkdir(parents=True, exist_ok=True)


def has_module(name: str) -> bool:
    return importlib.util.find_spec(name) is not None


for module in ["pytesseract", "easyocr", "paddleocr", "cv2", "layoutparser"]:
    print(f"{module:14s}: {has_module(module)}")
print("tesseract bin :", shutil.which("tesseract"))

/Users/judy/Desktop/github/mo_du_yeon/Advanced-RAG-Multimodal-AI-Agent/venv/bin/python
pytesseract   : True
easyocr       : True
paddleocr     : True
cv2           : True
layoutparser  : True
tesseract bin : /opt/homebrew/bin/tesseract


In [ ]:
from __future__ import annotations

import importlib.util
import sys
import os
import re
import shutil
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from PIL import Image, ImageDraw

print(sys.executable)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
IMG_DIR = PROJECT_ROOT / "docs" / "imgs"

# 외부 라이브러리가 홈 디렉터리에 캐시를 만들지 않도록 프로젝트 내부로 고정합니다.
os.environ.setdefault("PADDLE_PDX_CACHE_HOME", str(PROJECT_ROOT / "data" / "paddlex_cache"))
os.environ.setdefault("MPLCONFIGDIR", str(PROJECT_ROOT / "data" / "matplotlib_cache"))
os.environ.setdefault("XDG_CACHE_HOME", str(PROJECT_ROOT / "data" / "xdg_cache"))
for key in ["PADDLE_PDX_CACHE_HOME", "MPLCONFIGDIR", "XDG_CACHE_HOME"]:
    Path(os.environ[key]).mkdir(parents=True, exist_ok=True)


def has_module(name: str) -> bool:
    return importlib.util.find_spec(name) is not None


for module in ["pytesseract", "easyocr", "paddleocr", "cv2", "layoutparser"]:
    print(f"{module:14s}: {has_module(module)}")
print("tesseract bin :", shutil.which("tesseract"))

/Users/judy/Desktop/github/mo_du_yeon/Advanced-RAG-Multimodal-AI-Agent/venv/bin/python
pytesseract   : True
easyocr       : True
paddleocr     : True
cv2           : True
layoutparser  : True
tesseract bin : /opt/homebrew/bin/tesseract


## 1. Document Loader로 PDF 불러오기

In [ ]:
loader = PyPDFLoader(str(PDF_PATH))
documents = loader.load()

print(f"Loaded pages: {len(documents)}")
print(documents[0].metadata)
print(documents[0].page_content[:500])

## 2. Text Splitter로 문서 청크 분할하기

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=700, # 700, 500
    chunk_overlap=150, # 50, 100, 150
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = text_splitter.split_documents(documents)

print(f"Chunks: {len(chunks)}")
print(chunks[0].metadata)
print(chunks[0].page_content[:700])

## 3. PDF 텍스트 추출 상태 확인하기

현재 Naive RAG는 이미지를 직접 보는 방식이 아니라 PDF에서 추출된 텍스트를 검색합니다. 그래서 그래프 안 숫자가 텍스트로 추출되는지 먼저 확인합니다.

In [ ]:
for page_number, doc in enumerate(documents, start=1):
    text = doc.page_content
    if "분기 별 매출" in text or "파이낸셜 플랫폼" in text:
        print(f"\n--- page {page_number} ---")
        print(text[:1800])

## 4. Embedding 적용하기

In [ ]:
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY가 없습니다. .env 파일을 확인해주세요.")

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

sample_vector = embeddings.embed_query("네이버 2026년 1분기 파이낸셜 플랫폼 매출")
print(f"Embedding dimension: {len(sample_vector)}")
print(sample_vector[:5])

## 5. Vector DB 생성하기

Chroma DB는 `data/chroma_naver_1q26`에 저장됩니다.

In [ ]:
vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name=COLLECTION_NAME,
    persist_directory=str(DB_DIR),
)

if hasattr(vector_db, "persist"):
    vector_db.persist()

print(f"Vector DB saved to: {DB_DIR}")

## 6. Retriever로 관련 문서 검색하기

In [ ]:
retriever = vector_db.as_retriever(search_kwargs={"k": 4})

query = "파이낸셜 플랫폼 막대 그래프에서 분기 별 매출을 분석해줘"
retrieved_docs = retriever.invoke(query)

for i, doc in enumerate(retrieved_docs, start=1):
    page = doc.metadata.get("page")
    page_label = f"p.{page + 1}" if isinstance(page, int) else "page unknown"
    print(f"\n--- retrieved {i}: {page_label} ---")
    print(doc.page_content[:900])

## 7. 검색된 문서를 기반으로 LLM 답변 생성하기

In [ ]:
def format_context(docs):
    formatted = []
    for doc in docs:
        page = doc.metadata.get("page")
        page_label = f"p.{page + 1}" if isinstance(page, int) else "page unknown"
        formatted.append(f"[{page_label}]\n{doc.page_content}")
    return "\n\n".join(formatted)


prompt = ChatPromptTemplate.from_template(
    """너는 네이버 2026년 1분기 실적발표 PDF를 근거로 답하는 분석 assistant야.
주어진 context에 있는 내용만 사용해서 한국어로 답해.
숫자, 기간, 사업부 이름은 원문과 맞게 정확히 적고, 근거 페이지를 함께 표시해.
특히 단위가 '십억 원', '조 원', '%' 중 무엇인지 혼동하지 말고 원문 단위를 그대로 밝혀.
context에 없는 내용이면 모른다고 말해.

Question:
{question}

Context:
{context}
"""
)

llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

chain = (
    {
        "context": retriever | format_context,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

answer = chain.invoke(query)
print(answer)

## 8. Naive RAG 한계 테스트 질문

아래 질문들은 검색은 잘 되더라도 단위 변환, 표/그래프 범위 차이, 계산 정확도에서 약점이 드러날 수 있습니다.

In [ ]:
test_questions = [
    "파이낸셜 플랫폼 그래프에서 4Q24부터 1Q26까지 분기별 매출 증가액과 증가율을 계산하고, 원문 단위가 십억 원인지 억 원인지 확인해서 답해줘.",
    "파이낸셜 플랫폼 그래프에서 매출과 결제액(TPV)을 구분해서 설명해줘. 두 지표의 단위가 어떻게 다른지도 알려줘.",
    "5페이지와 6페이지의 분기별 매출 그래프를 비교해서 네이버 플랫폼과 파이낸셜 플랫폼 중 1Q26 QoQ 성장률이 더 높은 쪽을 알려줘.",
]

for question in test_questions:
    print("=" * 80)
    print(question)
    print("-" * 80)
    print(chain.invoke(question))
    print()